In [1]:
import sys
# sys.path.append('../..')
import json
import time

import bp3d
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [12]:
# sdge dev
url='https://wifire-data.sdsc.edu:4001/api'
# url = 'https://bp3d.nrp-nautilus.io'
c = bp3d.Client(url=url)

In [13]:
fuel = c.fuel(ftype='uniform', xlen=600, ylen=600, density=0.7, height=1)
fuel

In [14]:
ignitions = []
# for x in ['../data/ignite_aerial.dat',
#           '../data/ignite_longfireline_outwards.dat',
#           '../data/ignite_strip_southwards.dat',
#           '../data/ignite_longfireline_inwards.dat',
#           '../data/ignite_strip_northwards.dat']:

for x in ['../data/ignite_longfireline_outwards.dat']:
    ignitions.append(c.ignition(dat=x, perc=100))

ConnectTimeout: HTTPSConnectionPool(host='wifire-data.sdsc.edu', port=4001): Max retries exceeded with url: /api/v1/import/ignition (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7fb60b273590>, 'Connection to wifire-data.sdsc.edu timed out. (connect timeout=None)'))

In [ ]:
ignitions

In [26]:
ens = c.ensemble(cartesian=True,
                 fuel=fuel,
                 sim_time=600,
                 wind_speed=[1, 2, 3, 5, 7.5, 10, 15],
                 wind_direction=list(range(230, 340, 10)),
                 surface_moisture=[0.05, 0.1, 0.15],
                 output={
                     'steps_fire': 1,
                     'steps_wind': 1,
                     'energy_atmos': True,
                     'fire_energy': True,
                     'fuels_moist': True,
                 },
                 topo={
                     'total_startup_iters': 0
                 },
                 ignition=ignitions)
len(ens.df)

1155

In [ ]:
ens.execute()

In [28]:
ens.df

,sim_time,fuel,wind_speed,wind_direction,surface_moisture,output,topo,ignition,run_id,st_state,st_status,st_done
0,600,"{'xlen': 600, 'ylen': 600, 'density': 0.7, 'he...",1.0,230,0.05,"{'steps_fire': 1, 'steps_wind': 1, 'energy_atm...",{'total_startup_iters': 0},"{'perc': 100, 'dat': [' igntype= 4 ',...",80aa1808-adc0-42c1-88e5-abfe3cfc5c41,unknown,unknown,0
1,600,"{'xlen': 600, 'ylen': 600, 'density': 0.7, 'he...",1.0,230,0.05,"{'steps_fire': 1, 'steps_wind': 1, 'energy_atm...",{'total_startup_iters': 0},"{'perc': 100, 'dat': [' igntype= 5 ',...",d7a29471-89ef-4372-a39f-16f3f306edde,unknown,unknown,0
2,600,"{'xlen': 600, 'ylen': 600, 'density': 0.7, 'he...",1.0,230,0.05,"{'steps_fire': 1, 'steps_wind': 1, 'energy_atm...",{'total_startup_iters': 0},"{'perc': 100, 'dat': [' igntype= 5 ',...",85a4e12f-3016-4531-a062-fe5e6dca07eb,unknown,unknown,0
3,600,"{'xlen': 600, 'ylen': 600, 'density': 0.7, 'he...",1.0,230,0.05,"{'steps_fire': 1, 'steps_wind': 1, 'energy_atm...",{'total_startup_iters': 0},"{'perc': 100, 'dat': [' igntype= 5 ',...",dd17e766-419e-439f-91e4-876a48d600bf,unknown,unknown,0
4,600,"{'xlen': 600, 'ylen': 600, 'density': 0.7, 'he...",1.0,230,0.05,"{'steps_fire': 1, 'steps_wind': 1, 'energy_atm...",{'total_startup_iters': 0},"{'perc': 100, 'dat': [' igntype= 5 ',...",8f2da49a-9f67-4cce-b207-875c207b0ce9,unknown,unknown,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1150,600,"{'xlen': 600, 'ylen': 600, 'density': 0.7, 'he...",15.0,330,0.15,"{'steps_fire': 1, 'steps_wind': 1, 'energy_atm...",{'total_startup_iters': 0},"{'perc': 100, 'dat': [' igntype= 4 ',...",56b35589-2b95-402d-9786-e962bd6e4780,unknown,unknown,0
1151,600,"{'xlen': 600, 'ylen': 600, 'density': 0.7, 'he...",15.0,330,0.15,"{'steps_fire': 1, 'steps_wind': 1, 'energy_atm...",{'total_startup_iters': 0},"{'perc': 100, 'dat': [' igntype= 5 ',...",35075bfa-a1ed-4faa-bfd6-fc92c6e4954d,unknown,unknown,0
1152,600,"{'xlen': 600, 'ylen': 600, 'density': 0.7, 'he...",15.0,330,0.15,"{'steps_fire': 1, 'steps_wind': 1, 'energy_atm...",{'total_startup_iters': 0},"{'perc': 100, 'dat': [' igntype= 5 ',...",8ba41ffa-3e6a-4ff7-a1c4-2b9c97a97d77,unknown,unknown,0
1153,600,"{'xlen': 600, 'ylen': 600, 'density': 0.7, 'he...",15.0,330,0.15,"{'steps_fire': 1, 'steps_wind': 1, 'energy_atm...",{'total_startup_iters': 0},"{'perc': 100, 'dat': [' igntype= 5 ',...",c1e8ee75-c0d8-4d0f-bd79-8282d3b7960a,unknown,unknown,0


In [48]:
#ens.status()
ens.complete(timeout=60000, show_pbar=True)

100%|██████████| 100/100 [00:00<00:00, 174.61it/s]


In [49]:
ens_df = ens.df
print('FAILURE simulations: ' + str(len(ens_df[ens_df.st_state == 'FAILURE'])))
print('RUNNING simulations: ' + str(len(ens_df[ens_df.st_state == 'RUNNING'])))
print('OUTPUTS simulations: ' + str(len(ens_df[ens_df.st_state == 'OUTPUTS'])))
print('SUCCESS simulations: ' + str(len(ens_df[ens_df.st_state == 'SUCCESS'])))

FAILURE simulations: 0
RUNNING simulations: 0
OUTPUTS simulations: 0
SUCCESS simulations: 1155


In [53]:
ens.save('uniform-pgml.bp3d.json', overwrite=True)